# 알약 객체 검출 - YOLOv8n

2026-10-23 개인 프로젝트. AI 도구의 코드 작성·오류 대응 지원을 받아 Colab에서 실행했습니다.

실행 순서: 아래 셀을 위에서 아래로 실행합니다. Google Drive에 제공 데이터가 필요하며 첫 셀의 ROOT를 실제 위치에 맞추세요. T4 GPU를 사용합니다.

학습 232장 중 186장 학습 / 46장 검증, 56클래스, 15 epochs. Kaggle 점수 0.10093(사용자 제공). 내부 검증과 대회 지표는 다릅니다. 추가 실험은 수행하지 않았습니다.


## 1. Drive 연결과 데이터 확인

In [3]:
from google.colab import drive
from pathlib import Path
import torch
import json

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/코드잇/sprint_ai_project1_data")

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
print("데이터 폴더:", ROOT.exists())

if ROOT.exists():
    for folder in sorted(ROOT.iterdir()):
        if folder.is_dir():
            count = 0
            examples = []
            for p in folder.rglob("*"):
                if p.is_file():
                    count += 1
                    if len(examples) < 2:
                        examples.append(str(p.relative_to(ROOT)))
            print(f"\n{folder.name}: {count:,}개")
            print("파일 예시:", examples)

    annotation = next(
        (ROOT / "train_annotations").rglob("*.json"), None
    )
    if annotation:
        with open(annotation, encoding="utf-8-sig") as f:
            data = json.load(f)
        print("\nJSON 샘플:")
        print(json.dumps(data, ensure_ascii=False, indent=2)[:6000])
else:
    print("Drive 내 실제 폴더 위치를 확인해야 합니다.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
데이터 폴더: True

test_images: 287개
파일 예시: ['test_images/1371.png', 'test_images/1369.png']

train_annotations: 763개
파일 예시: ['train_annotations/K-003544-004543-016548-027993_json/K-016548/K-003544-004543-016548-027993_0_2_0_2_70_000_200.json', 'train_annotations/K-003544-004543-016548-027993_json/K-016548/K-003544-004543-016548-027993_0_2_0_2_75_000_200.json']

train_images: 232개
파일 예시: ['train_images/K-001900-016548-019607-029451_0_2_0_2_70_000_200.png', 'train_images/K-003544-004543-012247-016548_0_2_0_2_70_000_200.png']

JSON 샘플:
{
  "images": [
    {
      "file_name": "K-003544-004543-016548-027993_0_2_0_2_70_000_200.png",
      "width": 976,
      "height": 1280,
      "imgfile": "K-003544-004543-016548-027993_0_2_0_2_70_000_200.png",
      "drug_N": "K-016548",
      "drug_S": "정상알약",
      "back_color": "연회색 배경",
      "drug_dir": "앞면",
    

## 2. JSON 병합과 정합성 검사

In [4]:
import json
from collections import defaultdict

image_files = {
    p.name: p
    for p in (ROOT / "train_images").rglob("*")
    if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"}
}

labels = defaultdict(list)
classes = {}
errors = []

for p in (ROOT / "train_annotations").rglob("*.json"):
    try:
        with open(p, encoding="utf-8-sig") as f:
            d = json.load(f)

        images_by_id = {
            im["id"]: im for im in d.get("images", [])
        }

        for c in d.get("categories", []):
            classes[int(c["id"])] = c.get("name", str(c["id"]))

        for ann in d.get("annotations", []):
            im = images_by_id.get(ann.get("image_id"))
            if im is None:
                errors.append(f"이미지 ID 불일치: {p.name}")
                continue
            name = im["file_name"].replace("\\", "/").split("/")[-1]
            labels[name].append(ann)
    except Exception as e:
        errors.append(f"{p.name}: {e}")

missing = sorted(set(labels) - set(image_files))
unlabeled = sorted(set(image_files) - set(labels))

print("학습 이미지:", len(image_files))
print("정답에 등장하는 이미지:", len(labels))
print("클래스:", len(classes))
print("이미지가 없는 정답:", len(missing))
print("정답이 없는 이미지:", len(unlabeled))
print("JSON 오류:", len(errors))
print("이미지당 정답 수:", dict(
    __import__("collections").Counter(len(v) for v in labels.values())
))
print("누락 이미지 예시:", missing[:3])
print("오류 예시:", errors[:3])

학습 이미지: 232
정답에 등장하는 이미지: 232
클래스: 56
이미지가 없는 정답: 0
정답이 없는 이미지: 0
JSON 오류: 0
이미지당 정답 수: {4: 74, 3: 151, 2: 7}
누락 이미지 예시: []
오류 예시: []


## 3. COCO → YOLO 변환 및 분할

In [5]:
!pip -q install ultralytics==8.4.184

import json, random, shutil, math
from pathlib import Path
from collections import Counter
from PIL import Image
import yaml

WORK = Path("/content/pill_dataset")
OUT = ROOT.parent / "pill_results"
OUT.mkdir(exist_ok=True)

# 원본 클래스 번호 → YOLO용 연속 번호
category_ids = sorted({
    int(a["category_id"])
    for anns in labels.values()
    for a in anns
})
class_map = {cid: i for i, cid in enumerate(category_ids)}

# 재현 가능한 학습 80% / 검증 20% 분할
names = sorted(image_files)
random.Random(42).shuffle(names)
n_val = max(1, round(len(names) * 0.2))
splits = {"val": names[:n_val], "train": names[n_val:]}

invalid = 0
duplicates = 0
class_counts = Counter()

for split, filenames in splits.items():
    img_dir = WORK / "images" / split
    lbl_dir = WORK / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for name in filenames:
        src = image_files[name]
        with Image.open(src) as im:
            w, h = im.size

        lines = []
        seen = set()

        for a in labels[name]:
            box = a.get("bbox")
            if not isinstance(box, list) or len(box) != 4:
                invalid += 1
                continue

            x, y, bw, bh = map(float, box)
            if not all(math.isfinite(v) for v in (x, y, bw, bh)):
                invalid += 1
                continue
            if bw <= 0 or bh <= 0:
                invalid += 1
                continue

            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(w, x + bw), min(h, y + bh)
            if x2 <= x1 or y2 <= y1:
                invalid += 1
                continue

            cid = int(a["category_id"])
            key = (cid, x1, y1, x2, y2)
            if key in seen:
                duplicates += 1
                continue
            seen.add(key)

            xc = (x1 + x2) / (2 * w)
            yc = (y1 + y2) / (2 * h)
            nw = (x2 - x1) / w
            nh = (y2 - y1) / h
            lines.append(
                f"{class_map[cid]} {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}"
            )
            class_counts[cid] += 1

        shutil.copy2(src, img_dir / name)
        (lbl_dir / f"{Path(name).stem}.txt").write_text(
            "\n".join(lines), encoding="utf-8"
        )

config = {
    "path": str(WORK),
    "train": "images/train",
    "val": "images/val",
    "names": [str(classes.get(cid, cid)) for cid in category_ids],
}
(WORK / "data.yaml").write_text(
    yaml.safe_dump(config, allow_unicode=True), encoding="utf-8"
)

metadata = {
    "category_ids": category_ids,
    "class_names": config["names"],
    "splits": splits,
    "invalid_boxes_removed": invalid,
    "duplicate_boxes_removed": duplicates,
    "class_counts": dict(class_counts),
}
(OUT / "dataset_info.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("변환 완료")
print("학습:", len(splits["train"]), "/ 검증:", len(splits["val"]))
print("클래스:", len(category_ids))
print("제외한 잘못된 박스:", invalid)
print("제외한 중복 박스:", duplicates)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 11.4 MB/s eta 0:00:00
변환 완료
학습: 186 / 검증: 46
클래스: 56
제외한 잘못된 박스: 1
제외한 중복 박스: 0


## 4. 학습과 결과 저장

In [6]:
from ultralytics import YOLO
import ultralytics
import torch

assert torch.cuda.is_available(), "런타임을 T4 GPU로 변경하세요."

model = YOLO("yolov8n.pt")

model.train(
    data=str(WORK / "data.yaml"),
    epochs=15,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    seed=42,
    project="/content/pill_runs",
    name="baseline",
    exist_ok=False,
    plots=True,
)

# 실제 생성된 경로에서 결과를 Drive로 보관
RUN = Path(model.trainer.save_dir)
SAVED_RUN = OUT / RUN.name
shutil.copytree(RUN, SAVED_RUN, dirs_exist_ok=True)

(OUT / "environment.txt").write_text(
    f"ultralytics=={ultralytics.__version__}\n"
    f"torch=={torch.__version__}\n",
    encoding="utf-8"
)

print("\n학습 완료 / 저장 위치:", SAVED_RUN)
print((RUN / "results.csv").read_text().splitlines()[-1])

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pill_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.

## 5. 테스트 예측과 제출 CSV

In [7]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import json

OUT = ROOT.parent / "pill_results"

with open(OUT / "dataset_info.json", encoding="utf-8") as f:
    info = json.load(f)

category_ids = info["category_ids"]

# 학습된 최적 모델
predictor = YOLO(str(SAVED_RUN / "weights" / "best.pt"))

test_files = sorted(
    [
        p for p in (ROOT / "test_images").rglob("*")
        if p.is_file() and p.suffix.lower() == ".png"
    ],
    key=lambda p: int(p.stem)
)

assert test_files, "테스트 이미지가 없습니다."
assert len({int(p.stem) for p in test_files}) == len(test_files), \
    "테스트 이미지 ID가 중복됩니다."

rows = []
empty_images = []

predictions = predictor.predict(
    source=[str(p) for p in test_files],
    imgsz=640,
    conf=0.01,
    iou=0.5,
    max_det=4,
    device=0,
    stream=True,
    verbose=False,
)

for result in predictions:
    image_id = int(Path(result.path).stem)
    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        empty_images.append(image_id)
        continue

    for xyxy, cls, score in zip(
        boxes.xyxy.cpu().tolist(),
        boxes.cls.cpu().tolist(),
        boxes.conf.cpu().tolist(),
    ):
        x1, y1, x2, y2 = xyxy
        rows.append({
            "annotation_id": len(rows) + 1,
            "image_id": image_id,
            "category_id": int(category_ids[int(cls)]),
            "bbox_x": x1,
            "bbox_y": y1,
            "bbox_w": x2 - x1,
            "bbox_h": y2 - y1,
            "score": score,
        })

columns = [
    "annotation_id", "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
]
submission = pd.DataFrame(rows, columns=columns)

assert not submission.empty, "예측 결과가 없습니다."
assert not submission.isna().any().any()
assert submission["annotation_id"].is_unique
assert (submission[["bbox_w", "bbox_h"]] > 0).all().all()
assert submission["score"].between(0, 1).all()

csv_path = OUT / "submission.csv"
submission.to_csv(csv_path, index=False)

print("테스트 이미지:", len(test_files))
print("예측 객체:", len(submission))
print("검출 없는 이미지:", len(empty_images))
print("저장 위치:", csv_path)
display(submission.head())

from google.colab import files
files.download(str(csv_path))

테스트 이미지: 287
예측 객체: 1148
검출 없는 이미지: 0
저장 위치: /content/drive/MyDrive/코드잇/pill_results/submission.csv


,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,1900,160.429077,255.281570,201.219299,120.500839,0.755442
1,2,1,27733,555.786011,70.900024,392.871338,408.119141,0.587449
2,3,1,35206,591.456238,664.788940,254.033447,484.908203,0.359601
3,4,1,18357,173.470215,743.596313,180.992249,284.429199,0.234278
4,5,10,16548,101.378326,806.322144,241.807465,238.377808,0.958234


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>